# Diffusion Lab — PyTorch / Diffusers quickstart

Скачайте **только этот .ipynb**, откройте в Jupyter или Colab и выполните ячейки.
Пакет встроен в первую ячейку; локальная папка проекта и GitHub не нужны.
Нужны Python 3.11+ и интернет для установки недостающих зависимостей.

Работа идёт через обычные `model`, `scheduler`, `DataLoader`, `optimizer` и
ручной цикл обучения. Ни сборка, ни training loop не требуют `.fit`-интерфейса.
Менять компонент можно обычной заменой Python-объекта или тела функции.


In [ ]:
#@title Установить Diffusion Lab и зависимости
# This cell contains the small, verified Python wheel; no repository URL is needed.
import base64
import hashlib
import importlib
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("Diffusion Lab requires Python 3.11 or newer")
_diffusion_lab_wheel_data = (
    "..."
)
_diffusion_lab_wheel_bytes = base64.b64decode(_diffusion_lab_wheel_data)
if hashlib.sha256(_diffusion_lab_wheel_bytes).hexdigest() != "7712dffde405ccfda4775d86e3e789e69cc1617844c150972ef34ca6536a5f5b":
    raise RuntimeError("Embedded package checksum mismatch")
if importlib.util.find_spec("pip") is None:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
with tempfile.TemporaryDirectory(prefix="diffusion-lab-install-") as _install_dir:
    _wheel_path = Path(_install_dir) / "diffusion_lab-0.2.0-py3-none-any.whl"
    _wheel_path.write_bytes(_diffusion_lab_wheel_bytes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--disable-pip-version-check", str(_wheel_path) + "[interactive]"])
    # Replace only the tiny local package, even when its version is already installed.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--disable-pip-version-check", "--force-reinstall", "--no-deps",
                           str(_wheel_path)])
importlib.invalidate_caches()
print("Diffusion Lab is installed in:", sys.executable)


## 1. Данные, модель, scheduler, pipeline

`PointDenoiser` — torch-модель с Diffusers `ModelMixin` / `ConfigMixin`.
Scheduler — настоящий `DDIMScheduler`; `ResearchDiffusionPipeline` занимается
генерацией. Размерность пространства выводится из данных, `hidden_dim` задаёт
ширину сети. Для GPU поменяйте `DEVICE` на `cuda`.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from diffusers import DDIMScheduler
from diffusion_lab import PointDenoiser, ResearchDiffusionPipeline
from diffusion_lab.environments import make_environment
from diffusion_lab.visualization import plot_dataset
from diffusion_lab.metrics import evaluate_samples

DEVICE = "cpu"  # "cuda", "mps"
torch.manual_seed(42)
torch.set_num_threads(4)
dataset = make_environment("rotating_rays", n_samples=1024, n_classes=4, seed=42)
model = PointDenoiser(
    sample_shape=dataset.sample_shape,
    hidden_dim=64, num_layers=3, backbone="mlp",
    conditioner="embedding", num_classes=dataset.num_classes,
).to(DEVICE)
scheduler = DDIMScheduler(
    num_train_timesteps=100, beta_schedule="squaredcos_cap_v2",
    prediction_type="v_prediction", clip_sample=False, timestep_spacing="trailing",
)
pipeline = ResearchDiffusionPipeline(model, scheduler)
loader_generator = torch.Generator().manual_seed(43)
loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=loader_generator)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = F.mse_loss
print("Data:", dataset.samples.shape)
plot_dataset(dataset.samples, dataset.labels, title="Rotating rays")
plt.show()
plt.close()


## 2. Обычный цикл обучения

Это весь шаг: взять batch, зашумить через scheduler, вызвать model,
вычислить выбранный loss и сделать `backward/step`. Можно менять sampling t,
condition dropout, loss, clipping, optimizer и schedule прямо здесь.
Target обязан соответствовать `scheduler.config.prediction_type`.


In [ ]:
def train_torch(model, scheduler, loader, optimizer, epochs, loss_fn=F.mse_loss,
                condition_dropout=0.15):
    device = model.device
    history = []
    model.train()
    for epoch in range(epochs):
        total_loss, examples = 0.0, 0
        for batch in loader:
            clean = batch["samples"].to(device)
            conditions = batch["conditions"].to(device)
            noise = torch.randn_like(clean)
            timesteps = torch.randint(scheduler.config.num_train_timesteps,
                                      (len(clean),), device=device)
            noisy = scheduler.add_noise(clean, noise, timesteps)
            prediction_type = scheduler.config.prediction_type
            if prediction_type == "epsilon":
                target = noise
            elif prediction_type == "v_prediction":
                target = scheduler.get_velocity(clean, noise, timesteps)
            elif prediction_type == "sample":
                target = clean
            else:
                raise ValueError(prediction_type)
            keep_condition = torch.rand(len(clean), device=device) >= condition_dropout
            optimizer.zero_grad(set_to_none=True)
            prediction = model(noisy, timesteps, conditions=conditions,
                               condition_mask=keep_condition).sample
            loss = loss_fn(prediction, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(clean)
            examples += len(clean)
        history.append({"epoch": epoch + 1, "loss": total_loss / examples})
    return history

history = train_torch(model, scheduler, loader, optimizer, epochs=20, loss_fn=loss_fn)
plt.plot([row["epoch"] for row in history], [row["loss"] for row in history])
plt.xlabel("Epoch")
plt.ylabel("Training objective")
plt.grid(alpha=0.2)
plt.show()
plt.close()
print(history[-1])


## 3. Генерация в стиле Diffusers

Пайплайн использует ту же обученную модель. CFG требует условного dropout в
цикле выше. `guidance_scale=1` — обычное условное предсказание.
Короткое обучение иллюстрирует интерфейс; качество оценивается отдельно.


In [ ]:
conditions = torch.arange(256) % dataset.num_classes
output = pipeline(
    batch_size=256, conditions=conditions,
    num_inference_steps=50, guidance_scale=1.5,
    generator=torch.Generator().manual_seed(42), record_history=True,
)
plot_dataset(output.samples, conditions, title="Generated rotating rays")
plt.show()
plt.close()
print(evaluate_samples(dataset, output.samples, conditions))

class_two = pipeline(batch_size=128, conditions=2, num_inference_steps=50,
                     generator=torch.Generator().manual_seed(123))
plot_dataset(class_two.samples, title="Condition = 2")
plt.show()
plt.close()


## 4. Поле и потенциальная проекция

Query-точки уже считаются x_t. `score`, `epsilon` и `denoising` имеют разные
значения и масштабы. Потенциал восстанавливается в соглашении F=−∇U;
проверяйте residual/curl, чтобы понять, можно ли трактовать его как точную энергию.
Для flow вместо `score` нужен `denoising` или `velocity`.


In [ ]:
from diffusion_lab.visualization import (Grid, evaluate_field, plot_vector_field,
    recover_potential, plot_potential)

grid = Grid.from_bounds([(-1.2, 1.2), (-1.2, 1.2)], resolution=32)
field = evaluate_field(lambda points: pipeline.vector_field(
    points, timestep=35, conditions=2, kind="score"), grid)
plot_vector_field(grid, field, samples=dataset.samples,
                  stride=2, streamlines=True, title="Score, class=2, t=35")
plt.show()
plt.close()
potential = recover_potential(grid, field)
print("Residual:", potential.relative_residual, "curl:", potential.curl_rms)
print(potential.message)
plot_potential(grid, potential)
plt.show()
plt.close()


## 5. Diffusers save/load + torch optimizer checkpoint

`save_pretrained` сохраняет модель/scheduler, `torch.save` — optimizer и RNG.
Чтобы продолжить в новой сессии, перенесите сохранённую папку вместе с checkpoint,
воссоздайте тот же dataset/DataLoader и выполните ячейку восстановления.
Для пользовательской модели импортируйте её класс перед загрузкой или передайте
экземпляр в `from_pretrained(..., model=your_model)`.


In [ ]:
from pathlib import Path
saved = Path("outputs/torch_quickstart")
pipeline.save_pretrained(saved / "pipeline")
torch.save({
    "optimizer": optimizer.state_dict(),
    "completed_epochs": len(history),
    "torch_rng": torch.get_rng_state(),
    "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    "loader_rng": loader_generator.get_state(),
}, saved / "optimizer.pt")

restored = ResearchDiffusionPipeline.from_pretrained(
    saved / "pipeline", local_files_only=True, low_cpu_mem_usage=False,
).to(DEVICE)
restored_optimizer = torch.optim.AdamW(restored.model.parameters(), lr=1e-3)
checkpoint = torch.load(saved / "optimizer.pt", map_location="cpu", weights_only=True)
restored_optimizer.load_state_dict(checkpoint["optimizer"])
torch.set_rng_state(checkpoint["torch_rng"])
loader_generator.set_state(checkpoint["loader_rng"])
if torch.cuda.is_available() and checkpoint["cuda_rng"]:
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
continued = train_torch(restored.model, restored.scheduler, loader,
                        restored_optimizer, epochs=2, loss_fn=loss_fn)
print("Total completed epochs:", checkpoint["completed_epochs"] + len(continued))


## Пустой Jupyter и другие компоненты

Для нового notebook можно скачать wheel рядом с ним и выполнить:
```python
%pip install "./diffusion_lab-0.2.0-py3-none-any.whl[interactive]"
```
После установки используйте обычные `PointDenoiser`, Diffusers scheduler,
`torch.optim`, `DataLoader` и собственный training loop, как выше.
В этом стартовом файле wheel уже встроен в первую ячейку.

`04_change_components.ipynb` показывает **обученные** примеры замены backbone,
своей модели, MSE/Huber/своего loss, one-hot/embedding/своего conditioner,
optimizer, размерностей, rectified flow и navigation.
`DiffusionTrainer` и `create_experiment` доступны как необязательные утилиты.
